# Fetal Head Ultrasound Segmentation with YOLO26

**Multi-class instance segmentation of Brain, CSP, and LV**

Uses repository: [mfazrinizar/fetal-head-segmentation](https://github.com/mfazrinizar/fetal-head-segmentation)

## 1. Environment Setup

In [ ]:
import os
import sys
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
print(f"Running on: {'Kaggle' if IS_KAGGLE else 'Local'}")

In [ ]:
if IS_KAGGLE:
    import kagglehub
    print("Setting up Kaggle environment...")
    path = kagglehub.dataset_download("mfazrinizar/fetal-head-segmentation-yolo-splitted")

    print("Path to dataset files:", path)
    # Clone repo (skip if exists)
    REPO_DIR = Path('/kaggle/working/fetal-head-segmentation')
    if not REPO_DIR.exists():
        !git clone https://github.com/mfazrinizar/fetal-head-segmentation.git {REPO_DIR}
    else:
        print(f"Repo already exists: {REPO_DIR}")
    
    # Find dataset in read-only input (search recursively for train folder)
    import subprocess
    result = subprocess.run(['find', '/kaggle/input', '-type', 'd', '-name', 'train', '-maxdepth', '10'], 
                           capture_output=True, text=True)
    train_dirs = [p for p in result.stdout.strip().split('\n') if p and (Path(p) / 'images').exists()]
    
    if train_dirs:
        INPUT_DATA_DIR = Path(train_dirs[0]).parent
        print(f"Found dataset: {INPUT_DATA_DIR}")
    else:
        # Show what's available
        print("ERROR: Could not find dataset with train/images structure")
        !find /kaggle/input -maxdepth 5 -type d 2>/dev/null | head -20
        raise FileNotFoundError("Dataset not found in /kaggle/input")
    
    # Setup writable directories
    DATA_DIR = REPO_DIR / 'data'
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR = Path('/kaggle/working/results')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
else:
    REPO_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_DIR = REPO_DIR / 'data'
    INPUT_DATA_DIR = DATA_DIR  # Local: data is in repo
    OUTPUT_DIR = REPO_DIR / 'results'

print(f"Repo: {REPO_DIR}")
print(f"Data dir (writable): {DATA_DIR}")
if IS_KAGGLE:
    print(f"Input data (read-only): {INPUT_DATA_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
if IS_KAGGLE:
    print("Installing YOLO26...")
    !pip install -q git+https://github.com/mfazrinizar/ultralytics.git
    !cd {REPO_DIR} && pip install -q -e .
else:
    print("Using local environment")

In [ ]:
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / 'src'))
print("[OK] Python path configured")

## 2. Import Modules

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

from src.util.constants import (
    CLASS_NAMES, FETSAM_CLASS_WEIGHTS, EXPERIMENTS,
    TRAINING_HYPERPARAMS
)
from src.model.fetsam_loss_integration import (
    FetSAMBCEDiceLovaszLoss, patch_yolo_loss, unpatch_yolo_loss
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Create dataset.yaml pointing to read-only input
if IS_KAGGLE:
    import yaml
    
    dataset_config = {
        'path': str(INPUT_DATA_DIR),
        'train': 'train/images',
        'val': 'val/images',
        'test': 'test/images',
        'names': {0: 'Brain', 1: 'CSP', 2: 'LV'},
        'nc': 3
    }
    
    dataset_yaml = DATA_DIR / 'dataset.yaml'
    with open(dataset_yaml, 'w') as f:
        yaml.dump(dataset_config, f)
    
    print(f"[OK] Created dataset.yaml: {dataset_yaml}")
    print(f"     Points to: {INPUT_DATA_DIR}")
else:
    dataset_yaml = DATA_DIR / 'dataset.yaml'
    
    if not dataset_yaml.exists():
        raise FileNotFoundError(f"Dataset not found: {dataset_yaml}")
    
    print(f"[OK] Dataset: {dataset_yaml}")

# Verify splits exist
for split in ['train', 'val', 'test']:
    if IS_KAGGLE:
        split_dir = INPUT_DATA_DIR / split
    else:
        split_dir = DATA_DIR / split
    
    if split_dir.exists():
        n_img = len(list((split_dir / 'images').glob('*.png')))
        n_lbl = len(list((split_dir / 'labels').glob('*.txt')))
        print(f"  {split}: {n_img} images, {n_lbl} labels")
    else:
        print(f"  [WARN] {split} not found: {split_dir}")

**Dataset Setup for Kaggle:**

The dataset in `/kaggle/input` is READ-ONLY.

This notebook will:
1. Auto-detect your dataset location in `/kaggle/input`
2. Create a new `dataset.yaml` in `/kaggle/working/fetal-head-segmentation/data/`
3. Point it to the read-only input paths

Upload your dataset with any name - it will be auto-detected.

## 3. Configuration

In [ ]:
class Config:
    # Change EXPERIMENT to use class-balanced L model
    # Options: 'baseline', 'fetsam_full', 'fetsam_balanced_l', 'fetsam_balanced_m'
    MODEL_SIZE = 'l'  # Will be overridden by experiment config if specified
    EXPERIMENT = 'fetsam_balanced_l'  # Class-balanced YOLO26L
    EPOCHS = 300
    # Batch sizes for T4x2 (16GB each): L=16, M=32, N=128
    BATCH_SIZE = 16 if not IS_KAGGLE else 16  # Smaller for L model
    DEVICE = '0' if not IS_KAGGLE else '0,1'
    WORKERS = 8
    QUICK_TEST = False

cfg = Config()
cfg.REPO_DIR = REPO_DIR
cfg.DATA_DIR = DATA_DIR
cfg.OUTPUT_DIR = OUTPUT_DIR

# Get experiment config and override settings if specified
exp_cfg = EXPERIMENTS[cfg.EXPERIMENT]
if 'img_size' in exp_cfg:
    cfg.IMG_SIZE = exp_cfg['img_size']
else:
    cfg.IMG_SIZE = 640
if 'batch_size' in exp_cfg:
    cfg.BATCH_SIZE = exp_cfg['batch_size']
cfg.MODEL_SIZE = exp_cfg['model_size']

print(f"Experiment: {cfg.EXPERIMENT} - {exp_cfg['name']}")
print(f"Description: {exp_cfg['description']}")
print(f"Model: YOLO26{cfg.MODEL_SIZE}-seg")
print(f"Image size: {cfg.IMG_SIZE}")
print(f"Epochs: {cfg.EPOCHS}")
print(f"Batch size: {cfg.BATCH_SIZE}")
print(f"Device: {cfg.DEVICE}")
print(f"Available experiments: {list(EXPERIMENTS.keys())}")

## 4. Dataset Verification

In [ ]:
# Verify dataset.yaml exists (created in earlier cell on Kaggle)
dataset_yaml = cfg.DATA_DIR / 'dataset.yaml'

if not dataset_yaml.exists():
    print(f"[WARN] dataset.yaml not found at {dataset_yaml}")
    print("This should have been created earlier. Check IS_KAGGLE detection.")
else:
    print(f"[OK] Dataset: {dataset_yaml}")

# On Kaggle, splits are in read-only INPUT_DATA_DIR; locally in DATA_DIR
splits_base = INPUT_DATA_DIR if IS_KAGGLE else cfg.DATA_DIR
for split in ['train', 'val', 'test']:
    split_dir = splits_base / split
    if split_dir.exists():
        n_img = len(list((split_dir / 'images').glob('*.png')))
        n_lbl = len(list((split_dir / 'labels').glob('*.txt')))
        print(f"  {split}: {n_img} images, {n_lbl} labels")
    else:
        print(f"  [WARN] {split} not found: {split_dir}")

In [ ]:
def viz_samples(data_dir, n=4):
    imgs = list((data_dir / 'train' / 'images').glob('*.png'))[:n]
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    if n == 1: axes = [axes]
    colors = {'Brain': (255, 0, 0), 'CSP': (0, 255, 0), 'LV': (0, 0, 255)}
    
    for ax, img_path in zip(axes, imgs):
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        lbl_path = data_dir / 'train' / 'labels' / (img_path.stem + '.txt')
        
        if lbl_path.exists():
            h, w = img.shape[:2]
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    cls_id = int(parts[0])
                    coords = [float(x) for x in parts[1:]]
                    pts = np.array([[int(coords[i]*w), int(coords[i+1]*h)] 
                                   for i in range(0, len(coords), 2)], np.int32)
                    if len(pts) > 0:
                        cv2.polylines(img, [pts], True, list(colors.values())[cls_id], 2)
        
        ax.imshow(img)
        ax.set_title(img_path.name[:15])
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

# On Kaggle, samples are in read-only INPUT_DATA_DIR
viz_samples(INPUT_DATA_DIR if IS_KAGGLE else cfg.DATA_DIR)

## 5. FetSAM Loss

In [ ]:
print("Testing FetSAM loss...")
loss_fn = FetSAMBCEDiceLovaszLoss(
    weight_bce=0.25, weight_dice=0.5, weight_lovasz=0.25,
    class_weights=FETSAM_CLASS_WEIGHTS
)

pred = torch.randn(2, 3, 64, 64, requires_grad=True)
target = torch.randint(0, 2, (2, 3, 64, 64)).float()
loss = loss_fn(pred, target)
loss.backward()

print(f"Loss: {loss.item():.4f}")
print(f"Gradient: {'[OK]' if pred.grad is not None else '[FAIL]'}")
print(f"Weights: {FETSAM_CLASS_WEIGHTS}")

## 6. Training

In [ ]:
exp_cfg = EXPERIMENTS[cfg.EXPERIMENT]
print(f"Config: {exp_cfg}")

if cfg.QUICK_TEST:
    cfg.EPOCHS = 1
    cfg.BATCH_SIZE = 8
    print("[WARNING] Quick test: 1 epoch")

use_fetsam = exp_cfg['custom_loss']
if use_fetsam:
    print("Patching YOLO with FetSAM loss...")
    patch_yolo_loss()

model = YOLO(f"yolo26{cfg.MODEL_SIZE}-seg.yaml")

In [ ]:
# Offline Augmentation (FetSAM-style + Domain-Guided)
# Runs once before training - skips if already done
from src.preprocess.offline_augmentation import offline_augment_dataset, check_augmentation_status
from src.preprocess.domain_guided_augmentation import run_domain_guided_augmentation, check_domain_guided_status

aug_dir = DATA_DIR  # Writable directory

# Check experiment config for augmentation options
use_domain_guided = exp_cfg.get('use_domain_guided', False)
use_offline_aug = exp_cfg.get('use_offline_aug', False) or not use_domain_guided

# Domain-guided augmentation (context-preserving cut-paste)
if use_domain_guided:
    dg_status = check_domain_guided_status(aug_dir)
    print(f"Domain-guided augmentation: {'Done' if dg_status['done'] else 'Not done'}")
    if not dg_status['done']:
        multiplier = exp_cfg.get('domain_guided_multiplier', 1.5)
        print(f"Running domain-guided augmentation (multiplier={multiplier})...")
        result = run_domain_guided_augmentation(aug_dir, target_multiplier=multiplier)
        print(f"Result: {result}")

# Standard offline augmentation (whole-image transforms)
if use_offline_aug:
    aug_status = check_augmentation_status(aug_dir)
    print(f"Offline augmentation: {'Done' if aug_status['augmented'] else 'Not done'}")
    if not aug_status['augmented']:
        multiplier = exp_cfg.get('offline_aug_multiplier', 2.0)
        print(f"Running offline augmentation (multiplier={multiplier})...")
        result = offline_augment_dataset(aug_dir, target_multiplier=multiplier, prioritize_minority=True)
        print(f"Result: {result['status']}")
    else:
        print(f"Dataset at {aug_status['info'].get('total_after', 'unknown')} images")

# Show final counts
from collections import Counter
from pathlib import Path
labels_dir = aug_dir / "train" / "labels"
counts = Counter()
for lf in labels_dir.glob("*.txt"):
    with open(lf) as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                counts[int(parts[0])] += 1
print(f"Final class distribution: Brain={counts[0]}, CSP={counts[1]}, LV={counts[2]}")

In [ ]:
results = model.train(
    data=str(dataset_yaml),
    epochs=cfg.EPOCHS,
    batch=cfg.BATCH_SIZE,
    imgsz=cfg.IMG_SIZE,
    device=cfg.DEVICE,
    workers=cfg.WORKERS,
    project=str(cfg.OUTPUT_DIR),
    name=cfg.EXPERIMENT,
    exist_ok=True,
    pretrained=True,
    optimizer='auto',
    lr0=0.0002,
    lrf=TRAINING_HYPERPARAMS['lrf'],
    weight_decay=TRAINING_HYPERPARAMS['weight_decay'],
    warmup_epochs=TRAINING_HYPERPARAMS['warmup_epochs'],
    cos_lr=TRAINING_HYPERPARAMS['cos_lr'],
    patience=TRAINING_HYPERPARAMS['patience'],
    seed=42,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
    **exp_cfg['augmentation']
)

if use_fetsam:
    unpatch_yolo_loss()
    
print("[OK] Training complete!")

## 7. Evaluation

In [ ]:
best_pt = cfg.OUTPUT_DIR / cfg.EXPERIMENT / 'weights' / 'best.pt'
if not best_pt.exists():
    best_pt = cfg.OUTPUT_DIR / cfg.EXPERIMENT / 'weights' / 'last.pt'

if best_pt.exists():
    print(f"[OK] Model: {best_pt}")
    model = YOLO(str(best_pt))
    
    val_res = model.val(data=str(dataset_yaml), split='val', batch=cfg.BATCH_SIZE,
                        imgsz=640, device=cfg.DEVICE, plots=True, save_json=True)
    
    print("\n" + "="*50)
    print("VALIDATION")
    print("="*50)
    print(f"Box mAP@50: {val_res.box.map50:.4f}")
    print(f"Mask mAP@50: {val_res.seg.map50:.4f}")
    print(f"Mask mAP@50-95: {val_res.seg.map:.4f}")
    
    print("\nPer-class:")
    for i, name in enumerate(CLASS_NAMES):
        if hasattr(val_res.seg, 'ap50') and i < len(val_res.seg.ap50):
            print(f"  {name}: {val_res.seg.ap50[i]:.4f}")
else:
    print(f"[FAIL] Model not found: {best_pt}")

In [ ]:
if best_pt.exists():
    test_res = model.val(data=str(dataset_yaml), split='test', batch=cfg.BATCH_SIZE,
                         imgsz=640, device=cfg.DEVICE, plots=True, save_json=True)
    
    print("\n" + "="*50)
    print("TEST")
    print("="*50)
    print(f"Box mAP@50: {test_res.box.map50:.4f}")
    print(f"Mask mAP@50: {test_res.seg.map50:.4f}")
    print(f"Mask mAP@50-95: {test_res.seg.map:.4f}")

## 8. Visualization

In [ ]:
if best_pt.exists():
    test_base = INPUT_DATA_DIR if IS_KAGGLE else cfg.DATA_DIR
    test_imgs = list((test_base / 'test' / 'images').glob('*.png'))[:6]
    preds = model.predict(source=[str(i) for i in test_imgs], conf=0.25, 
                         iou=0.7, save=False, verbose=False)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    colors = {'Brain': (1,0,0,0.4), 'CSP': (0,1,0,0.4), 'LV': (0,0,1,0.4)}
    
    for idx, (ax, res) in enumerate(zip(axes, preds)):
        img = cv2.cvtColor(res.orig_img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        
        if res.masks is not None:
            for mask, cls in zip(res.masks.data, res.boxes.cls):
                m = cv2.resize(mask.cpu().numpy(), (img.shape[1], img.shape[0]))
                overlay = np.zeros((*img.shape[:2], 4))
                overlay[m > 0.5] = colors[CLASS_NAMES[int(cls)]]
                ax.imshow(overlay)
        
        ax.set_title(f"Test {idx+1}")
        ax.axis('off')
    
    from matplotlib.patches import Patch
    leg = [Patch(facecolor=c[:3], alpha=0.5, label=n) for n, c in colors.items()]
    fig.legend(handles=leg, loc='lower center', ncol=3)
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / 'predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Training Curves

In [ ]:
import pandas as pd

csv_path = cfg.OUTPUT_DIR / cfg.EXPERIMENT / 'results.csv'
if csv_path.exists():
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    ax = axes[0, 0]
    for col in ['train/box_loss', 'train/seg_loss', 'train/cls_loss']:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=col.split('/')[-1])
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('Training Losses')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[0, 1]
    for col in ['metrics/mAP50(B)', 'metrics/mAP50-95(B)']:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=col.split('(')[0].split('/')[-1])
    ax.set_xlabel('Epoch')
    ax.set_title('Detection mAP')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1, 0]
    for col in ['metrics/mAP50(M)', 'metrics/mAP50-95(M)']:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=col.split('(')[0].split('/')[-1])
    ax.set_xlabel('Epoch')
    ax.set_title('Segmentation mAP')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    ax = axes[1, 1]
    for col in ['metrics/precision(M)', 'metrics/recall(M)']:
        if col in df.columns:
            ax.plot(df['epoch'], df[col], label=col.split('(')[0].split('/')[-1])
    ax.set_xlabel('Epoch')
    ax.set_title('Precision/Recall')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(cfg.OUTPUT_DIR / cfg.EXPERIMENT / 'curves.png', dpi=150)
    plt.show()

## 10. Summary

In [ ]:
print("="*60)
print("SUMMARY")
print("="*60)
print(f"Experiment: {cfg.EXPERIMENT}")
print(f"Model: YOLO26{cfg.MODEL_SIZE}-seg")
print(f"Classes: {CLASS_NAMES}")
print(f"FetSAM Loss: {EXPERIMENTS[cfg.EXPERIMENT]['custom_loss']}")
print(f"FetSAM Aug: {EXPERIMENTS[cfg.EXPERIMENT]['use_fetsam_aug']}")
print(f"Class Weights: {FETSAM_CLASS_WEIGHTS}")
print(f"Output: {cfg.OUTPUT_DIR / cfg.EXPERIMENT}")

if best_pt.exists():
    print("\n[OK] Complete")